# 04 — LoRA Fine-Tuning

Phase 8-9. **Requires a CUDA GPU.** Run the sanity check first.

> `train_model.py` refuses to start a full run if baseline results are
> missing — the experimental order cannot be reversed by accident.


In [ ]:
# --- Colab setup (skip if running locally) ---
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('casting-defect-vlm'):
        # Replace with your repository URL, or upload the folder to Colab.
        raise SystemExit('Upload the casting-defect-vlm project folder to Colab first.')
    %cd casting-defect-vlm
    !pip install -q -r requirements.txt

sys.path.insert(0, os.path.abspath('..' if os.path.basename(os.getcwd())=='notebooks' else '.'))
print('python', sys.version.split()[0], '| colab:', IN_COLAB)


In [ ]:
import torch
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    print('VRAM (GB)      :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
else:
    print('WARNING: no CUDA GPU. Baseline and training need one.')


## 1. Confirm the baseline exists


In [ ]:
from pathlib import Path
assert Path('results/baseline/baseline_results.csv').exists(), \
    'Run notebook 02 (the baseline) first — that is the whole point of the experiment.'
print('Baseline found. Proceeding.')


## 2. Training configuration


In [ ]:
import yaml
cfg = yaml.safe_load(open('config/config.yaml'))
print(yaml.dump({'model': cfg['model'], 'training': cfg['training'], 'lora': cfg['lora']}, sort_keys=False))


## 3. Sanity check (Phase 9)

8 examples, 2 steps. Verifies images load, the processor works, labels are
masked, forward/backward passes run, LoRA actually attached, the loss is
finite, and the adapter saves.


In [ ]:
!python scripts/train_model.py --sanity-check


## 4. Full training run

If you hit CUDA OOM, in `config/config.yaml`: lower `max_pixels`, keep
`batch_size: 1` and raise `gradient_accumulation_steps`, ensure
`gradient_checkpointing: true` and `load_in_4bit: true`. Document any
change you make — do not silently alter the methodology.


In [ ]:
!python scripts/train_model.py


## 5. Training summary


In [ ]:
import json
s = json.loads(Path('results/training/logs/training_summary.json').read_text())
for k in ['training_loss','global_step','trainable_params','total_params','trainable_pct']:
    print(f'{k:<22}{s[k]}')


## 6. Training curves


In [ ]:
from IPython.display import Image, display
p = Path('results/training/training_curves.png')
display(Image(str(p))) if p.exists() else print('no curves produced')


## 7. Verify the adapter reloads


In [ ]:
adapter = Path('results/training/final_model')
print('adapter dir exists:', adapter.exists())
print(sorted(f.name for f in adapter.iterdir())[:10])
